# Chapter 7: build the cross reference table and count unmatched keys

Section 7.4: the same pilsner malt sits under four codes. One item row and a cross reference row for each
source resolve it. Any code with no match goes to quarantine, not to a guess.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

xref = k.load("xref")
item = k.load("item").set_index("item_key")
print(xref[xref.entity == "item"].assign(item=lambda d: d.surrogate_key.map(item.item_name)).to_string(index=False))

Every code each source actually used in the slice.

In [ ]:
seen = pd.concat([
    pd.DataFrame({"source_system": "erp", "source_key": k.load("bronze_erp_goods_movement").item_code.astype(str)}),
    pd.DataFrame({"source_system": "silo_log", "source_key": k.load("bronze_silolog_draw").malt_code}),
    pd.DataFrame({"source_system": "brew_sheet", "source_key": k.load("bronze_brewsheet_brew").malt_name}),
    pd.DataFrame({"source_system": "lims", "source_key": k.load("bronze_lims_material").material_code}),
]).drop_duplicates()
current = xref[xref.valid_to.isna()]
joined = seen.merge(current, on=["source_system", "source_key"], how="left")
unmatched = joined[joined.surrogate_key.isna()]
print(f"{len(seen)} codes seen, {len(unmatched)} unmatched")
print(unmatched[["source_system", "source_key"]].assign(shown_as=lambda d: d.source_key.map(repr)).to_string(index=False))

Units: one lager brew draws 5,300 kg of grist. The factor table turns each source's unit into kg.

In [ ]:
factor = k.load("unit_factor").set_index("from_unit")
for unit, amount in [("bag50", 106), ("t", 5.3), ("kg", 5300)]:
    print(f"{amount} {unit} = {amount * factor.factor[unit] + factor.intercept[unit]:,.0f} kg")
print(f"one hL holds {1 / factor.factor['can330']:.2f} cans; a pallet of 3,600 cans is "
      f"{3600 * factor.factor['can330']:.2f} hL")

Time: a brew that mashes in at 02:15 on Friday 20 March 2026 IST belongs to Thursday's brew day
(06:00 to 06:00 IST, from the shift table).

In [ ]:
shift = k.load("shift")
local = pd.Timestamp("2026-03-20 02:15")
utc = local - pd.Timedelta(hours=5, minutes=30)
day_start = pd.Timedelta(hours=int(shift[shift.shift_name == "A"].start_time.iloc[0][:2]))
brew_day = (local - day_start).date()
print(f"UTC {utc:%A %d %B %H:%M}; IST calendar {local:%A %d %B}; brew day {brew_day:%A %d %B}")